# 03 — Vektorisasi, Pembangunan Index & Visualisasi Embedding

Notebook ini menampilkan tahap di mana **model Transformer (IndoSBERT) berperan paling penting**:
mengubah setiap chunk pasal menjadi vektor angka yang merepresentasikan maknanya,
menyimpannya ke index yang bisa dicari cepat, lalu memvisualisasikan hasilnya.

| Bagian | Isi |
|---|---|
| **A** | BM25 Index — tokenisasi & build |
| **B** | IndoSBERT + FAISS — encode, build, simpan |
| **C** | Visualisasi embedding — PCA, t-SNE, heatmap similarity |

**Prasyarat:** sudah menjalankan `01_pengolahan_pdf` (ada `chunks.jsonl`) dan `scripts/00_download_model.py`.

In [ ]:
import sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
from src.config import load_config, resolve_path
from src.bm25_retriever import BM25Retriever, tokenize
from src.semantic_retriever import SemanticRetriever

cfg = load_config()
processed  = resolve_path(cfg['paths']['processed_dir'])
index_dir  = resolve_path(cfg['paths']['index_dir'])
models_dir = resolve_path(cfg['paths']['models_dir'])

chunks = [json.loads(l) for l in open(processed / 'chunks.jsonl', encoding='utf-8')]
doc_ids = [c['id']   for c in chunks]
texts   = [c['text'] for c in chunks]
domains = [c['id'].split('_')[0] for c in chunks]

print(f'Total chunk: {len(chunks)}')
print(f'Contoh ID  : {doc_ids[:3]}')

---
## Bagian A — BM25 Index

BM25 bekerja **tanpa Transformer** — murni berbasis statistik kata.
Ditampilkan dulu agar kita bisa membandingkan pendekatannya dengan IndoSBERT.

### A1. Tokenisasi — Dari Teks ke Daftar Kata

In [ ]:
for c in chunks[5:8]:
    tokens = tokenize(c['text'])
    print(f"ID    : {c['id']}")
    print(f"Teks  : {c['text'][:120]}...")
    print(f"Token : {tokens[:15]} ... ({len(tokens)} token total)")
    print()

### A2. Bangun & Uji BM25 Index

In [ ]:
bm25 = BM25Retriever(
    k1=cfg['bm25']['k1'],
    b=cfg['bm25']['b'],
    use_sastrawi=(cfg['bm25']['tokenizer'] == 'sastrawi'),
)
t0 = time.time()
bm25.build(doc_ids, texts)
print(f'BM25 dibangun dalam {time.time()-t0:.2f} detik | {len(bm25.doc_ids)} dokumen')

query_demo = 'uang pesangon pemutusan hubungan kerja'
print(f'\nContoh pencarian: "{query_demo}"')
for rank, (doc_id, score) in enumerate(bm25.search(query_demo, top_k=3), 1):
    idx = doc_ids.index(doc_id)
    print(f'  #{rank}  {doc_id:<40}  skor={score:.4f}')
    print(f'       {texts[idx][:90]}...')

---
## Bagian B — IndoSBERT + FAISS Index

Di sinilah **Transformer berperan**. IndoSBERT mengubah teks menjadi vektor 768 dimensi
yang merepresentasikan makna semantik, bukan sekadar frekuensi kata.

### B1. Muat Model IndoSBERT

In [ ]:
sem = SemanticRetriever(
    model_name=cfg['embedding']['model_name'],
    batch_size=cfg['embedding']['batch_size'],
    device=cfg['embedding']['device'],
    cache_folder=str(models_dir),
)
print(f'Model : {sem.model_name}  |  Device : {sem.device}')
sem._load_model()
n_params = sum(p.numel() for p in sem.model._first_module().parameters())
print(f'Model siap. Parameter: {n_params:,}')

### B2. Apa itu Embedding?

Setiap teks diubah menjadi **vektor 768 angka**. Teks yang bermakna mirip menghasilkan vektor yang "dekat" satu sama lain.

In [ ]:
contoh_teks = [
    'berapa pesangon kalau dipecat?',
    'uang pesangon pemutusan hubungan kerja',
    'hak anak atas pendidikan',
]
emb_contoh = sem.encode(contoh_teks)
print(f'Bentuk matriks: {emb_contoh.shape}  (3 teks × 768 dimensi)\n')
for i, teks in enumerate(contoh_teks):
    print(f'Teks {i+1}: "{teks}"')
    print(f'  10 dim pertama: {emb_contoh[i, :10].round(4).tolist()}\n')

### B3. Encode Seluruh Corpus & Bangun FAISS Index

In [ ]:
import faiss

print(f'Encoding {len(texts)} chunk...')
t0 = time.time()
semua_emb = sem.encode(texts)
durasi = time.time() - t0
print(f'Selesai: {durasi:.1f} detik | {semua_emb.shape} | {semua_emb.nbytes/1024**2:.1f} MB')

dim = semua_emb.shape[1]
index_faiss = faiss.IndexFlatIP(dim)
index_faiss.add(semua_emb)
sem.index   = index_faiss
sem.doc_ids = doc_ids
print(f'\nFAISS index: {index_faiss.ntotal} vektor × {index_faiss.d} dim  |  IndexFlatIP (exact search)')

### B4. Perbandingan BM25 vs IndoSBERT pada Query Bahasa Sehari-hari

In [ ]:
query = 'berapa pesangon kalau dipecat?'
hasil_bm  = bm25.search(query, top_k=5)
hasil_sem = sem.search(query, top_k=5)

df_cmp = pd.DataFrame({
    'BM25 (kata)':       [f"{d}  [{s:.4f}]" for d, s in hasil_bm],
    'IndoSBERT (makna)': [f"{d}  [{s:.4f}]" for d, s in hasil_sem],
})
df_cmp.index = [f'#{i+1}' for i in range(len(df_cmp))]
print(f'Query: "{query}"\n')
print(df_cmp.to_string())

### B5. Simpan Kedua Index ke Disk

In [ ]:
import os
index_dir.mkdir(parents=True, exist_ok=True)

bm25_path  = index_dir / 'bm25.pkl'
faiss_path = index_dir / 'faiss'
bm25.save(bm25_path)
sem.save(faiss_path)

faiss_file = faiss_path.with_suffix('.faiss')
print(f'BM25  : {bm25_path}  ({os.path.getsize(bm25_path)/1024:.0f} KB)')
print(f'FAISS : {faiss_file}  ({os.path.getsize(faiss_file)/1024**2:.1f} MB)')

---
## Bagian C — Visualisasi Embedding

Vektor 768 dimensi tidak bisa dilihat langsung. Kita reduksi ke 2 dimensi agar bisa
diplot — dan melihat **apakah pasal bertema sama benar-benar saling berdekatan**.
Jika ya, itu bukti bahwa IndoSBERT memahami makna, bukan sekadar kata.

### C1. Reduksi Dimensi dengan PCA (768 → 2 dimensi)

PCA cepat dan mempertahankan **struktur global** — baik untuk melihat pemisahan antar domain.

In [ ]:
from sklearn.decomposition import PCA

PALETTE = {
    'KETENAGAKERJAAN': '#4C72B0',
    'KONSUMEN':        '#55A868',
    'ITE':             '#C44E52',
    'ANAK':            '#8172B3',
}

def scatter_plot(coords, title):
    plt.figure(figsize=(9, 7))
    for dom, color in PALETTE.items():
        mask = [d == dom for d in domains]
        plt.scatter(coords[mask, 0], coords[mask, 1],
                    s=14, alpha=0.6, c=color, label=dom.capitalize())
    plt.title(title); plt.legend(title='Domain Hukum')
    plt.tight_layout(); plt.show()

pca_model  = PCA(n_components=2, random_state=42)
coords_pca = pca_model.fit_transform(semua_emb)
scatter_plot(coords_pca, 'Embedding IndoSBERT (PCA) — diwarnai per domain')

### C2. Reduksi Dimensi dengan t-SNE (struktur lokal lebih jelas)

t-SNE lebih baik menampilkan **klaster** — pasal sejenis akan terlihat berkumpul.
Proses ini membutuhkan waktu sekitar 10–30 detik.

In [ ]:
from sklearn.manifold import TSNE

coords_tsne = TSNE(
    n_components=2, perplexity=30, init='pca',
    learning_rate='auto', random_state=42
).fit_transform(semua_emb)

scatter_plot(coords_tsne, 'Embedding IndoSBERT (t-SNE) — pasal mengelompok berdasarkan makna')

print('Interpretasi: titik warna sama saling berkumpul = IndoSBERT menempatkan')
print('pasal bertema serupa berdekatan di ruang vektor, tanpa diberi tahu labelnya.')

### C3. Query Bahasa Awam Mendarat di Klaster yang Tepat

Kita encode query sehari-hari lalu proyeksikan ke ruang PCA yang sama.
Apakah query tentang ketenagakerjaan mendarat di dekat klaster ketenagakerjaan?

In [ ]:
queries_demo = [
    'berapa pesangon kalau dipecat?',
    'ditipu saat belanja online',
    'anak menjadi korban kekerasan',
]
q_emb = sem.encode(queries_demo)
q_pca = pca_model.transform(q_emb)

plt.figure(figsize=(9, 7))
for dom, color in PALETTE.items():
    mask = [d == dom for d in domains]
    plt.scatter(coords_pca[mask, 0], coords_pca[mask, 1],
                s=12, alpha=0.35, c=color, label=dom.capitalize())
plt.scatter(q_pca[:, 0], q_pca[:, 1],
            s=220, c='black', marker='*', zorder=5, label='QUERY')
for (x, y), q in zip(q_pca, queries_demo):
    plt.annotate(q, (x, y), fontsize=8, xytext=(6, 6), textcoords='offset points')
plt.title('Query (bintang ★) mendarat dekat domain yang relevan')
plt.legend(); plt.tight_layout(); plt.show()

### C4. Heatmap Cosine Similarity — Makna Serupa, Kata Berbeda

Pasangan kalimat yang **bermakna sama** seharusnya berskor tinggi meski kata-katanya berbeda.
Ini yang tidak bisa dilakukan BM25.

In [ ]:
kalimat = [
    'pekerja dipecat dari perusahaan',
    'pemutusan hubungan kerja terhadap buruh',
    'konsumen membeli barang yang rusak',
    'pembeli menerima produk cacat',
]
E = sem.encode(kalimat)
sim = E @ E.T

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim, cmap='viridis', vmin=0, vmax=1)
ax.set_xticks(range(len(kalimat)))
ax.set_yticks(range(len(kalimat)))
ax.set_xticklabels([k[:22] for k in kalimat], rotation=40, ha='right', fontsize=8)
ax.set_yticklabels([k[:22] for k in kalimat], fontsize=8)
for i in range(len(kalimat)):
    for j in range(len(kalimat)):
        ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center',
                color='white' if sim[i,j] < 0.7 else 'black', fontsize=10)
plt.colorbar(im, label='Cosine Similarity')
ax.set_title('Kemiripan Semantik antar-kalimat (IndoSBERT)')
plt.tight_layout(); plt.show()
print('Kalimat 1↔2 ("dipecat" vs "PHK") dan 3↔4 ("barang rusak" vs "produk cacat")')
print('berskor TINGGI meski kata berbeda — inilah keunggulan Transformer atas BM25.')

---
## Ringkasan: BM25 vs IndoSBERT

In [ ]:
ringkasan = pd.DataFrame([
    {'Aspek': 'Cara kerja',    'BM25': 'Hitung frekuensi kata cocok',      'IndoSBERT': 'Encode teks → vektor makna'},
    {'Aspek': 'Representasi',  'BM25': 'Daftar token + frekuensi',          'IndoSBERT': f'Vektor {dim} dimensi (float32)'},
    {'Aspek': 'Waktu build',   'BM25': '< 1 detik',                         'IndoSBERT': f'{durasi:.0f} detik'},
    {'Aspek': 'Ukuran index',  'BM25': f'{os.path.getsize(bm25_path)/1024:.0f} KB', 'IndoSBERT': f'{os.path.getsize(faiss_file)/1024**2:.1f} MB'},
    {'Aspek': 'Kekuatan',      'BM25': 'Istilah hukum eksak, nomor pasal', 'IndoSBERT': 'Query bahasa sehari-hari'},
    {'Aspek': 'Kelemahan',     'BM25': 'Gagal bila kata beda, makna sama', 'IndoSBERT': 'Kurang presisi untuk istilah sangat spesifik'},
]).set_index('Aspek')
print(ringkasan.to_string())
print()
print('→ Keduanya digabung dengan RRF di notebook 04_demo_retrieval.')